# Stage 10 - Pledged-securities 1D trend status (live refresh, full universe)

**Corrected design:** a purely read-only version can go stale the same way the
credit-spread screener's already-documented gap works - if a symbol flipped
after the last fetch, the saved report keeps showing the old direction/age.
This version genuinely refreshes every symbol before reporting status.

**Second correction - a real regression found while building the Stage 11
pledged-trend-following ranking:** the original version of this cell called
`run_all_instruments(..., force_refresh=True)` with a short 390-day window,
which does a full REPLACE of each symbol's saved raw data, not an update.
Every routine live-screener run was silently throwing away notebook 12's deep
multi-year history and replacing it with just that short window - after
repeated runs, every symbol's saved history had collapsed to ~1 year, and
UP-closed-flip counts per symbol fell to 1-8, nowhere near usable for a
ranking. Fixed by switching to `incremental=True`
(`process_instrument_incremental()` in `pledged_1d_pipeline.py`): it EXTENDS
existing saved history with just the recent catch-up window instead of
replacing it outright, so routine checks no longer erode the deep history
notebook 12 built. `from_date` is no longer meaningful in this mode (each
symbol's own last-saved date drives its fetch window) - only `to_date` and
`delay_seconds` still apply.

**Rate limit hit three times in a row** even after the chunk-pacing fix in
`fetch_data.py` and a 1-second between-symbol delay - close-to-the-limit
pacing (~0.35-0.4s spacing, right at the documented 3 req/sec ceiling) isn't
leaving enough real-world margin for network jitter, and repeated failures in
the same session may also mean Kite briefly throttles further after tripping
the limit once. `delay_seconds=1.0` between symbols - real headroom, not the
minimum. If this still fails, it's worth waiting a minute or two before
retrying, in case of a lingering cooldown from the earlier attempts.

## Column reference

**Is the trend intact?**

| Column | Meaning |
|---|---|
| `direction` | The currently OPEN 1D Supertrend flip |
| `trend_age_days` | Calendar days since that flip |
| `spot` · `flip_level` | Latest close, and the Supertrend level that would reverse the trend |
| `buffer_to_flip_pct` | Distance between them. **Small = about to invalidate.** The most actionable number here for a holder deciding whether to stay pledged |
| `moved_since_entry_pct` | Signed *for* the trend: positive = working, negative = underwater |

**Does the signal actually work on this instrument?** *(new)*

| Column | Meaning |
|---|---|
| `dir_flips` | Closed flips on the currently open side. Under 10, the edge columns are blank |
| `dir_win_rate` | % of those that moved the way the signal called |
| `dir_median_move` | Median favourable move. Negative = the typical flip goes the *wrong* way |
| `signal_works_here` | `dir_win_rate >= 50` **and** `dir_median_move > 0` |
| `opp_win_rate` · `opp_median_move` | The other direction, for contrast only |

> **Why this matters on this universe.** Measured across 100 instruments: UP flips
> win 54.1% (+0.9% mean median move, 68% show an edge). DOWN flips win **42.4%**
> with a **−1.3%** mean median move and only **14%** show an edge — the median DOWN
> flip ends with price *higher* than where it flipped. Since a DOWN flip is what
> would prompt action on a pledged holding, most DOWN flips here are noise.
> **27 of 100 instruments have no edge in either direction.**

**Risk / event context**

| Column | Meaning |
|---|---|
| `worst_day` | Largest single close-to-close day against the position, 4yr |
| `entry_day_move` | Move on the day this flip was born |
| `born_on_outlier_day` | `abs(entry_day_move) > 4%` — a flip created by a one-day collapse is a different event, and the duration stats come from ordinary flips |

**Duration context** — descriptive only

| Column | Meaning |
|---|---|
| `median_duration` | This instrument's typical 1D flip length |
| `past_median` · `past_75th_pct` · `past_90th_pct` | Has the open trend outlived that checkpoint? |
| `runway_left_vs_median` · `_p75` · `_p90` | `checkpoint − age` in signed days; negative = past it |

> `reversion_test.py` and `exhaustion_test.py` both found that crossing a duration
> checkpoint does **not** predict a reversal. Not a warning, not an exit signal.

**Pledging metadata & score** *(from `pledged_securities.py` and notebook 18)*

| Column | Meaning |
|---|---|
| `security_type` · `etf_category` | Equity or ETF, and the ETF theme |
| `directly_fno_tradable` | Whether an NSE F&O contract exists on this symbol itself (62 of 154) |
| `pledged_trend_score` | UP-trend-following rank from notebook 18. Blank = thin sample, not a low score |
| `thin_sample` | Fewer than 15 UP-closed flips, so unscored |
| `outlier_reliance_flag` · `gap_risk_flag` | Score depends on a few outlier wins / high gap history |

*All statistics are in-sample and descriptive — what the signal HAS done, not what it will do.*


In [1]:
import sys
sys.path.append("..")

from datetime import datetime, timedelta
from src.auth import get_kite
from src.instruments import build_nse_token_lookup
from src.pledged_securities import load_pledged_universe
from src.pledged_1d_pipeline import run_all_instruments

kite = get_kite()
token_lookup = build_nse_token_lookup(kite)
universe = load_pledged_universe()
symbol_list = universe["symbol"].tolist()

# incremental=True - EXTENDS each symbol's existing saved history with just
# the recent catch-up window (per-symbol, derived from its own last-saved
# date) instead of replacing it outright. from_date is unused in this mode,
# kept only because run_all_instruments still accepts it positionally.
#
# delay_seconds bumped well past the 3 req/sec limit's minimum spacing
# (~0.33s) - three failed attempts in a row means it's worth real headroom
# over cutting it close, even though it makes the full run slower.
results = run_all_instruments(
    kite, token_lookup, symbol_list,
    from_date=None,
    to_date=datetime.now(),
    incremental=True,
    delay_seconds=1.0
)
results["status"].value_counts()

[1/154] ABBOTINDIA — processing...
Fetching 2025-07-03 to 2026-08-02...
Fetching 2026-08-03 to 2026-08-12...
Saved 2873 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ABBOTINDIA_day.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/pledged_1d/ABBOTINDIA_full_report.xlsx
[1/154] ABBOTINDIA — done
[2/154] ACC — processing...
Fetching 2025-07-03 to 2026-08-02...
Fetching 2026-08-03 to 2026-08-12...
Saved 2873 candles to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/data/raw/ACC_day.csv
Saved full report to /Users/hgmac/Library/CloudStorage/OneDrive-Personal/1 Files/2_harsh/1_harsh_work/3_trading/Quant/exports/pledged_1d/ACC_full_report.xlsx
[2/154] ACC — done
[3/154] AGARWALEYE — processing...
Fetching 2025-07-03 to 2026-08-02...
Fetching 2026-08-03 to 2026-08-12...
Saved 375 candles to /Use

status
SUCCESS    140
FAILED      14
Name: count, dtype: int64

In [2]:
from src.pledged_1d_screener import check_trend_status
import pandas as pd

results = [check_trend_status(s, universe_df=universe) for s in symbol_list]
results = [r for r in results if r is not None]
status = pd.DataFrame(results).sort_values("trend_age_days")
print("SORTED BY: trend_age_days ascending — sets the order the Shares/ETFs tables inherit")
print(f"{len(status)} of {len(symbol_list)} instruments have a current status")

No report found for BIRET - refresh it first
No report found for CMRGREEN - refresh it first
No report found for CORDELIA - refresh it first
No report found for EMBASSY - refresh it first
No report found for INDIGRID - refresh it first
No report found for IRBINVIT - refresh it first
No report found for KNACK - refresh it first
No report found for KRT - refresh it first
No report found for MINDSPACE - refresh it first
No report found for NSDL - refresh it first
No report found for NXST - refresh it first
No report found for PGINVIT - refresh it first
No report found for VAML - refresh it first
No report found for LIQUIDCASE - refresh it first
140 of 154 instruments have a current status


In [3]:
# Merge in score_for_pledged_trend_following()'s ranking (Stage 10/11) -
# answers "which of several same-day UP flips should I prioritize" instead
# of leaving it a guess. Run notebook 18 first if these files don't exist
# yet or look stale.
ranking = pd.read_excel("../exports/RANKING_pledged_trend_following.xlsx")
watchlist = pd.read_excel("../exports/WATCHLIST_pledged_trend_following.xlsx")

score_cols = ["symbol", "pledged_trend_score", "outlier_reliance_flag", "gap_risk_flag"]
ranking_slim = ranking[score_cols]

# Watchlist symbols have no score yet (below the 15 UP-closed-flip floor) -
# kept as NaN/blank rather than 0, so "no score" is never confused with
# "scored low". thin_sample flags which case applies.
watchlist_slim = watchlist[["symbol"]].copy()
for col in ["pledged_trend_score", "outlier_reliance_flag", "gap_risk_flag"]:
    watchlist_slim[col] = None
watchlist_slim["thin_sample"] = True

score_lookup = pd.concat([
    ranking_slim.assign(thin_sample=False),
    watchlist_slim
], ignore_index=True)

status = status.merge(score_lookup, on="symbol", how="left")
print(f"Scored: {status['pledged_trend_score'].notna().sum()} of {len(status)} — rest are thin-sample (< 15 UP-closed flips)")

Scored: 96 of 140 — rest are thin-sample (< 15 UP-closed flips)


## Shares

In [4]:
shares_status = status[status["security_type"] == "Equity"].reset_index(drop=True)
print("SORTED BY: trend_age_days ascending (inherited from `status`) — most recently flipped first")
shares_status


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,security_type,etf_category,directly_fno_tradable,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
0,COLPAL,DOWN,0,45.0,False,False,False,Equity,,True,55.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.50pp),NaN,False
1,GODREJCP,DOWN,0,52.0,False,False,False,Equity,,True,64.2,NaN,NaN,False
2,FORTIS,DOWN,0,48.0,False,False,False,Equity,,True,15.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 9.26pp),NaN,False
3,ALKEM,DOWN,0,48.0,False,False,False,Equity,,True,20.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.86pp),NaN,False
4,SANOFI,DOWN,0,56.0,False,False,False,Equity,,False,48.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.55pp),NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98,ASIANPAINT,UP,124,47.0,True,True,False,Equity,,True,72.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.42pp),NaN,False
99,RAINBOW,UP,127,71.0,True,True,False,Equity,,False,None,None,None,True
100,ANANDRATHI,UP,132,39.0,True,True,True,Equity,,False,None,None,None,True
101,BOSCHLTD,UP,132,45.5,True,True,True,Equity,,True,83.5,NaN,NaN,False


## ETFs

In [5]:
etf_status = status[status["security_type"] == "ETF"].reset_index(drop=True)
print("SORTED BY: trend_age_days ascending (inherited from `status`) — most recently flipped first")
etf_status


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,security_type,etf_category,directly_fno_tradable,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
0,PSUBNKBEES,UP,0,42.0,False,False,False,ETF,PSU Bank,False,36.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.18pp),⚠ HIGH GAP RISK (>10.0%),False
1,GOLDCASE,UP,2,100.0,False,False,False,ETF,Gold,False,None,None,None,True
2,MON100,UP,7,57.0,False,False,False,ETF,Nasdaq / International,False,79.7,NaN,⚠ HIGH GAP RISK (>10.0%),False
3,MAFANG,UP,8,32.5,False,False,False,ETF,Nasdaq / International,False,79.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.01pp),⚠ HIGH GAP RISK (>10.0%),False
4,GROWWRAIL,UP,9,57.0,False,False,False,ETF,Railways,False,None,None,None,True
5,DIVIDEND,UP,13,53.5,False,False,False,ETF,Dividend,False,None,None,None,True
6,QUAL30IETF,UP,14,52.5,False,False,False,ETF,Quality factor,False,None,None,None,True
7,ITBEES,UP,16,64.0,False,False,False,ETF,IT / Technology,False,20.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.49pp),⚠ HIGH GAP RISK (>10.0%),False
8,TOP100CASE,UP,37,144.5,False,False,False,ETF,Multicap / Total market,False,None,None,None,True
9,NIF100BEES,UP,37,53.5,False,False,False,ETF,Nifty 100,False,38.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.82pp),⚠ HIGH GAP RISK (>10.0%),False


## Top 25 UP-trending, ranked by pledged_trend_score (best first)

Sorted by score, not age — a fresh flip isn't automatically the best pick, and this is the view meant to answer that question directly when several instruments flip UP the same day. `trend_age_days` is still shown for context. Thin-sample names (score is blank) are included but sort to the bottom, not silently dropped.

### Shares

In [6]:
top25_shares_up = shares_status[shares_status["direction"] == "UP"].sort_values(
    "pledged_trend_score", ascending=False, na_position="last"
).head(25)
print("SORTED BY: pledged_trend_score descending, thin-sample (blank score) last")
top25_shares_up


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,security_type,etf_category,directly_fno_tradable,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
15,LT,UP,9,36.0,False,False,False,Equity,,True,88.4,NaN,NaN,False
66,SBIN,UP,61,39.5,True,False,False,Equity,,True,84.0,NaN,NaN,False
47,NESTLEIND,UP,37,47.5,False,False,False,Equity,,True,83.8,NaN,NaN,False
101,BOSCHLTD,UP,132,45.5,True,True,True,Equity,,True,83.5,NaN,NaN,False
62,TITAN,UP,57,41.0,True,False,False,Equity,,True,83.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.74pp),NaN,False
19,MARUTI,UP,12,39.0,False,False,False,Equity,,True,81.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.46pp),NaN,False
51,BRITANNIA,UP,41,49.0,False,False,False,Equity,,True,78.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.07pp),NaN,False
11,TATASTEEL,UP,7,43.5,False,False,False,Equity,,True,76.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.92pp),NaN,False
63,BAJAJFINSV,UP,57,51.0,True,False,False,Equity,,True,76.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 8.53pp),NaN,False
12,JSWSTEEL,UP,8,43.5,False,False,False,Equity,,True,75.8,NaN,NaN,False


### ETFs

In [7]:
top25_etf_up = etf_status[etf_status["direction"] == "UP"].sort_values(
    "pledged_trend_score", ascending=False, na_position="last"
).head(25)
print("SORTED BY: pledged_trend_score descending, thin-sample (blank score) last")
top25_etf_up


,symbol,direction,trend_age_days,median_duration,past_median,past_75th_pct,past_90th_pct,security_type,etf_category,directly_fno_tradable,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample
2,MON100,UP,7,57.0,False,False,False,ETF,Nasdaq / International,False,79.7,NaN,⚠ HIGH GAP RISK (>10.0%),False
3,MAFANG,UP,8,32.5,False,False,False,ETF,Nasdaq / International,False,79.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.01pp),⚠ HIGH GAP RISK (>10.0%),False
35,PVTBANIETF,UP,126,67.0,True,True,False,ETF,Private Bank,False,68.9,NaN,⚠ HIGH GAP RISK (>10.0%),False
29,CONSUMBEES,UP,119,91.0,True,False,False,ETF,FMCG / Consumption,False,64.7,NaN,⚠ HIGH GAP RISK (>10.0%),False
15,BANKBEES,UP,61,43.0,True,False,False,ETF,Bank Nifty / Banking,False,62.2,NaN,⚠ HIGH GAP RISK (>10.0%),False
34,JUNIORBEES,UP,124,61.5,True,True,False,ETF,Nifty Next 50,False,50.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 7.60pp),⚠ HIGH GAP RISK (>10.0%),False
11,INFRABEES,UP,56,55.5,True,False,False,ETF,Infra,False,42.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.82pp),⚠ HIGH GAP RISK (>10.0%),False
12,NIFTYBEES,UP,56,55.5,True,False,False,ETF,Nifty 50 / Broad Market,False,40.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.75pp),⚠ HIGH GAP RISK (>10.0%),False
9,NIF100BEES,UP,37,53.5,False,False,False,ETF,Nifty 100,False,38.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.82pp),⚠ HIGH GAP RISK (>10.0%),False
0,PSUBNKBEES,UP,0,42.0,False,False,False,ETF,PSU Bank,False,36.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.18pp),⚠ HIGH GAP RISK (>10.0%),False


## Today's fresh UP flips, ranked by score

The direct answer to "6 instruments flipped UP the same day, which do I buy" — filtered to only flips that happened in the most recent session (`trend_age_days <= 1`), sorted by `pledged_trend_score`. Empty is a normal, common result (this signal doesn't fire on most stocks most days) — it means nothing new to act on right now, not that something's broken.

In [8]:
fresh_up = status[(status["direction"] == "UP") & (status["trend_age_days"] <= 1)].sort_values(
    "pledged_trend_score", ascending=False, na_position="last"
)[["symbol", "security_type", "trend_age_days", "pledged_trend_score",
   "outlier_reliance_flag", "gap_risk_flag", "thin_sample", "etf_category"]]
print(f"{len(fresh_up)} fresh UP flip(s) today")
print("SORTED BY: pledged_trend_score descending, thin-sample last")
fresh_up


1 fresh UP flip(s) today


,symbol,security_type,trend_age_days,pledged_trend_score,outlier_reliance_flag,gap_risk_flag,thin_sample,etf_category
4,PSUBNKBEES,ETF,0,36.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.18pp),⚠ HIGH GAP RISK (>10.0%),False,PSU Bank
